# Lesson 03 - 具代理性的設計模式

在本課中，我們將探索建構有效 AI 代理人的三個基礎設計模式：

1. <strong>清晰的代理人指令</strong> — 撰寫精確且能定義角色的提示，引導代理人行為  
2. **使用 Pydantic 模型的結構化輸出** — 確保代理人回傳可預測且經過驗證的資料  
3. <strong>單一職責代理人</strong> — 設計專注且各司其職的代理人  

我們將這些模式套用在一個 <strong>旅遊目的地推薦系統</strong> 場景中，逐步建立一個能推薦目的地、檢查可用性及處理行程規劃的系統。


## 設置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 範例 1：清晰的代理指示

最具影響力的範例同時也是最簡單的：為你的代理編寫清晰、詳細的指示。

良好的指示定義：
- <strong>代理是誰</strong>（角色與語氣）
- <strong>應該做什麼</strong>（逐步職責）
- <strong>應該如何表現</strong>（限制與風格）

下面，我們建立一個旅遊禮賓代理，透過明確的指示塑造其所產生的每個回應。


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: 使用 Pydantic 模型的結構化輸出

自由格式的文字對於對話很有用，但下游系統需要結構化的數據。
通過將 **Pydantic 模型** 與 <strong>工具函數</strong> 配對，我們可以：

- 定義代理輸出的精確結構
- 自動驗證回應
- 可靠地將代理結果整合到應用邏輯中

我們還介紹了一個工具，該工具返回目的地詳細資料，讓代理基於真實數據來支持其建議。


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: 單一職責代理人

複雜任務透過分拆工作給多個專注的代理人，每個代理人只負責一項職責，可帶來好處：

- 一個了解地點和可用性的 <strong>目的地專家</strong>
- 一個負責航班、酒店和行程規劃的 <strong>物流策劃員</strong>

這反映了軟件工程中的<em>關注點分離</em>原則——每個代理人更容易獨立測試、維護和改進。


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Summary

在本課程中，我們將三個有代理特性的設計模式應用於旅遊推薦場景：

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | 預先定義角色、職責和限制 | 穩定、一致且符合品牌的代理行為 |
| **Structured Output** | 使用 Pydantic 模型作為回應格式 | 經驗證、機器可讀的結果 |
| **Single Responsibility** | 為每個代理分配一個專注的任務 | 更易於測試、維護與組合 |

這些模式可以自然地組合——您可以在單一職責的代理中結合清晰指示與結構化輸出，以構建穩健的生產環境系統。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
